In [1]:
import os
import sys
import geopandas as gpd
import rasterio
import rasterio.mask
import numpy as np
from shapely.geometry import Polygon
import pandas as pd
import math
import platform
import gc
import re
from multiprocessing import Pool

In [3]:
raster_directory = r"C:\Users\ruben.crespo\Downloads\vcs_layers"
# raster_directory = r"Z:\veg_c_storage_rawdata\vegetation_carbon_stock_global"

# Path to the shapefile containing the data on region polygons that are to be
# used in the aggregation.
# region_polygons_file = r"\\akif.internal\public\veg_c_storage_rawdata\wb_global_countries\2015_gaul_dataset_mod_2015_gaul_dataset_global_countries_1.shp"
region_polygons_file = r"C:\Users\ruben.crespo\Downloads\vcs_layers\2015_gaul_dataset_mod_2015_gaul_dataset_global_countries_1.shp"
# region_polygons_file = r"Z:\veg_c_storage_rawdata\wb_global_countries\2015_gaul_dataset_mod_2015_gaul_dataset_global_countries_1.shp"


# Name of the observable to aggregate. This will be used as a prefix for the
# temporary results filenames. A suffix "_YYYY.csv" will be appended to this
# name. The directory for the temporary exports is defined below.
observable_name = "vegetation-carbon-stock"

# Path for the temporal exports of the aggregation process after each raster
# processed.
# temp_export_dir = ".\vegetation.carbon.stock\tmp\\"
# temp_export_dir = "C:/Users/admin/Documents/01_Ruben_Scripts/im.nca.postprocessing/aggregation.region/vegetation.carbon.stock/tmp/"
temp_export_dir = r"C:/Users/ruben.crespo/Documents/02_Ruben_scripts/Python_codigo/im.nca.postprocessing/old_repo_structure/"
# temp_export_dir = ".\\vegetation.carbon.stock\tmp"

# Path to export the final dataset.
# export_path = "./tmp/vcs-aggregated-country-landcover.csv"
export_path = "C:/Users/ruben.crespo/Documents/02_Ruben_scripts/Python_codigo/im.nca.postprocessing/old_repo_structure/vcs-aggregated-country-landcover.csv"


In [4]:
def get_raster_data(path):
    """
    get_raster_data gets the addresses of all the raster files ("*.tif")
    stored in the directory specified by "path". Each raster file should
    correspond to a different year for the analysis.

    :param path: directory containing the raster files for the density observable
    to aggregate.
    :return: a list storing the addresses of all the raster files.
    """

    file_list = []
    for file in os.listdir(path):
        # Iterate over all the files in the specified directory.
        if ".tif" in file:
            # Process the file if it has a .tif format.
            if platform.system() == "Windows":
                address = os.path.join(path, file).replace("/","\\")
            else:
                address = os.path.join(path, file).replace("\\","/")
                #build the path according the OS running the script

            if address not in file_list:
                # Add the file address to the list if it had not been added before.
                file_list.append(address)
        else:
            pass

    return file_list


def load_region_polygons(file):
    """
    load_region_polygons loads a shapefile containing a polygon layer describing
    the regions to perform the aggregation.

    :param file: the address of the shapefile with the region polygons.
    :return: a GeoDataFrame with the data on the regions' polygons.
    """
    if platform.system() is "Windows":
        file = file.replace("/","\\")
    else:
        file = file.replace("\\","/")
        #build the path according the OS running the script

    gdf = gpd.read_file(file)
    return gdf


def export_to_csv(region_polygons, aggregated_observable, path):
    """
    export_to_csv joins the result of the aggregation process for each year with
    the regions and exports the final dataset in CSV format to the specified
    path.

    :param region_polygons: a GeoDataFrame storing the polygons corresponding to
    each region used to aggregate.
    :param aggregated_observable: a DataFrame storing the aggregated values of
    the observable.
    :param path: path for the export.
    :return: None. The function creates a file in the specified path with the
    final results of the aggregation.
    """

    # Create a DataFrame based on the regions GeoDataFrame and dropping
    # unnecessary information in order to keep only: the polygons' Id, region
    # codes, and administrative names.
    df_final = pd.DataFrame(region_polygons.drop(columns='geometry'))
    df_final = df_final.drop(["STATUS", "DISP_AREA", "ADM0_CODE", "STR0_YEAR", "EXP0_YEAR", "Shape_Leng", "ISO3166_1_", "ISO3166__1", "Shape_Le_1", "Shape_Area"], axis = 1)

    # Join the depurated regions DataFrame with the aggregated values.
    df_final = df_final.join(aggregated_observable)

    # Export the result to the specified path.
    df_final.to_csv(path)


###############################################################################
# Processing function.
###############################################################################


def area_of_pixel(pixel_size, center_lat):
    """
    area_of_pixel calculates the area, in hectares, of a wgs84 square raster
    tile given its latitude and side-length.
    This function is adapted from https://gis.stackexchange.com/a/288034.

    :param pixel_size: is the length of the pixel side in degrees.
    :param center_lat: is the latitude of the center of the pixel. This value
    +/- half the `pixel-size` must not exceed 90/-90 degrees latitude or an
    invalid area will be calculated.
    :return: the rel area in hectares of a square pixel of side length
    `pixel_size` whose center is at latitude `center_lat`.
    """

    a = 6378137  # meters
    b = 6356752.3142  # meters
    e = math.sqrt(1 - (b/a)**2)
    area_list = []
    for f in [center_lat+pixel_size/2, center_lat-pixel_size/2]:
        zm = 1 - e*math.sin(math.radians(f))
        zp = 1 + e*math.sin(math.radians(f))
        area_list.append(
            math.pi * b**2 * (
                math.log(zp/zm) / (2*e) +
                math.sin(math.radians(f)) / (zp*zm)))
    return (pixel_size / 360. * (area_list[0] - area_list[1])) * np.power(10.0,-4)


def split_and_aggregate(out_image, out_transform, pixel_size, width, height):
    """
    split_and_aggregate splits a raster in tiles of 1000x1000 pixels, performs
    the aggregation in each tile and accumulate the results to obtain the total
    value for all the region. The function is called when the raster mask
    corresponding to the region is too large in size (>3Gb).

    :param out_image: is the masked raster layer.
    :param out_transform: the Affine containing the transformation matrix with
    latitude and longitude values, resolution, etc.
    :param pixel_size: is the side length in degrees of each square raster tile.
    :param width: is the width of the masked layer.
    :param height: is the height of the masked layer.
    :return: the value of the aggregated observable at the region-level.
    """

    tilesize = 1000
    # The variable to accumulate the aggregated value of the observable.
    accumulated_agg_value = 0

    for i in range(0, width, tilesize): # Tilesize marks from where to where in width.
        for j in range(0, height, tilesize):
            # This is for the edge parts, so we don't get nodata from the borders.
            w0 = i # Start of the array.
            w_plus = min(i+tilesize, width) - i # Addition value.
            w1 = w0 + w_plus # End of the array.
            h0 = j
            h_plus = min(j+tilesize, height) - j
            h1 = h0 + h_plus

            tile_agg_value = aggregate_one_region(out_image, out_transform, pixel_size, w0, h0, w1, h1)

            accumulated_agg_value += tile_agg_value

    return accumulated_agg_value

def aggregate_one_region(out_image, out_transform, pixel_size, width_0, height_0, width_1, height_1):
    """
    aggregate_one_region performs the aggregation of the density observable in a
    given region supplied in the form of a masked raster of the observable.

    :param out_image: the masked raster layer to aggregate.
    :param out_transform: the Affine of the raster layer.
    :param pixel_size: the side length in degrees of each square raster tile.
    :param width_0: the starting value position of the width array
    :param height_0: the starting value position of the height array
    :param width_1: the end value position of the width array
    :param height_1: the end value position of the height array
    :return: the aggregated value of the observable in the specified region.
    """

    # Create a matrix of coordinates based on tile number.
    cols, rows = np.meshgrid(np.arange(width_0, width_1), np.arange(height_0, height_1))

    # Transform the tile number coordinates to real coordinates and extract only
    # latitude information.
    ys = rasterio.transform.xy(out_transform, rows, cols)[1] # [0] is xs
    latitudes = np.array(ys) # Cast the list of arrays to a 2D array for computational convenience.
    ys = cols = rows = None #empty the memory

    # Iterate over the latitudes matrix, calculate the area of each tile, and
    # store it in the real_raster_areas array.
    real_raster_areas = np.empty(np.shape(latitudes))
    for i, latitude_array in enumerate(latitudes):
        for j, latitude in enumerate(latitude_array):
            real_raster_areas[i,j] = area_of_pixel(pixel_size, latitude)

    # Calculate the total value in each tile: density * area = observable value
    # in the area.
    value = real_raster_areas * out_image[0,height_0:height_1,width_0:width_1] #I don't think np.transpose() is necesary
    out_image = None #empty the memory
    # Sum all the carbon stock values in the country treating NaNs as 0.0.
    aggregated_value = np.nansum(value)

    return aggregated_value

def get_geometry_grid(geodataframe, epsg):
    """
    get_geometry_grid creates a defined grid of the input territory extension.
    :geodataframe: the territory vector file ina gdf format.
    :epsg: the defined epsg of vector georeferenced data.
    :return: the grid as a geodataframe
    """
    
    #get the bounds of the territory
    xmin, ymin, xmax, ymax = geodataframe.total_bounds
    # define the size of the grid
    length = 10
    wide = 10
    # set the cols and rows
    cols = list(np.arange(xmin, xmax + wide, wide))
    rows = list(np.arange(ymin, ymax + length, length))
    # create a list of all the polygons containing the grid.
    polygons = []
    for x in cols[:-1]:
        for y in rows[:-1]:
            polygons.append(Polygon([(x,y), (x+wide, y), (x+wide, y+length), (x, y+length)]))
    # transform the polygon list into a Geoseries or Geodataframe.
    # grid = gpd.GeoSeries({'geometry':MultiPolygon(polygons)})
    grid = gpd.GeoDataFrame({'geometry':polygons}, crs=epsg)
    return grid


def extract_sublist(raster_files_list, value, position):
    """
    extract_sublist extracts a sublist filtering the main one according to the int value we
    want to filter from and its corresponding possition on the string values.
    :raster_files_list: a list containing the raster files of a directory
    :value_possition: the possition of the value on the string
    :position: possition of the int value in relation to all in the string
    :return: a filtered list
    """
    sublist_raster_files = []
    for item in raster_files_list:
        # if the value and the corresponding possition value of the list are equal, the path is appended
        if value == int(re.findall(r'\d+', item)[position]):
            sublist_raster_files.append(item)
        else:
            continue
        
    return sublist_raster_files

def parallel_argument_list(year_list, raster_files_list, region_polygons, temp_export_path):
    """
    parallel_argument_list creates a list containing different tupples from different input parameters
    :param year_list: the list of years to operate
    :param raster_files_list: a list containing the raster files of a directory
    :param region_polygons: a GeoDataFrame storing the polygons corresponding to
    each region used for the aggregation.
    :param temp_export_path: path to export the temporary results.
    :return: a DataFrame storing the aggregated vegetation carbon stocks at the
    region level for each year.
    
    """
    argument_list = []
    
    for year in year_list:
        list_year = extract_sublist(raster_files_list, year, 0)
        # we create a tuple
        argument_year = (list_year, region_polygons, temp_export_path)
        # append the tuple to the list
        argument_list.append(argument_year)
        
    return argument_list

def aggregate_density_observable_parallel_wrapper(args):
    return aggregate_density_observable(args[0],args[1],args[2])

def aggregate_density_observable(raster_files_list, region_polygons, temp_export_path):
    """
    aggregate_density_observable aggregates the density observable for all the
    raster files specified and inside the specified regions. The result of the
    aggregation is returned as a table and the result for each year/raster is
    progressively exported in CSV format.

    :param raster_files_list: a list containing the addresses of all the raster
    files that store the observable's data for each year.
    :param region_polygons: a GeoDataFrame storing the polygons corresponding to
    each region used for the aggregation.
    :param temp_export_path: path to export the temporary results.
    :return: a DataFrame storing the aggregated vegetation carbon stocks at the
    region level for each year.
    """

    # Final DataFrame will store the aggregated carbon stocks for each country and each year.
    aggregated_df = pd.DataFrame([])

    for file in raster_files_list[:]: # [10:]
        # Iterate over all the raster files' addresses and extract the year from the address.
        file_year = re.findall(r'\d+', file)[0]
        try:
           landcover_class = re.findall(r'\d+', file)[2]
        except:
            landcover_class = False

        print("Processing file {} corresponding to year {}.".format(file, file_year))

        # This list will store the results from the aggregation.
        aggregated_value_list = []

        with rasterio.open(file) as raster_file: # Load the raster file.

            gt = raster_file.transform # Get all the raster properties on a list.
            pixel_size = gt[0] # X size is stored in position 0, Y size is stored in position 4.

            error_countries_id = [] # Create a list for all encountered possible errors
            
            # create both counter if we want to tmp export resuls at reaching n number of countries
            # country_counter = 1
            # country_counter_iterator = 1

            for row_index, row in region_polygons.iterrows(): # gdf.loc[0:1].iterrows(): / gdf.loc(axis=0)[0:1] / df[df['column'].isin([1,2])]
                try:
                    # Iterate over the country polygons to progressively calculate the total carbon stock in each one of them.

                    geodf_row = gpd.GeoDataFrame(geometry=gpd.GeoSeries(row['geometry']), crs=4326) # This is the country's polygon geometry df.
                    # geo_row = gpd.GeoSeries(row['geometry']) # This is the country's polygon geometry.
                    """OPTION 1: split the raster"""
                    """this part is commented to avoid memory excess on raster"""
                    # # Masks the raster over the current country. The masking requires two outputs:
                    # # out_image: the array of the masked image. [z, y, x]
                    # # out_transform: the Affine containing the transformation matrix with lat / long values, resolution...
                    # out_image, out_transform = rasterio.mask.mask(raster_file, geo_row, crop=True)

                    # # Obtain the number of tiles in both directions.
                    # height = out_image.shape[1]
                    # width  = out_image.shape[2]

                    # Split the region / masked raster if it is too large to avoid memory
                    # errors. Else process the entire region.
                    # if out_image.nbytes > (1* 10**9):
                    #     print("Country {} exceeds 1Gb of memory, splitting the array in tiles of 1000x1000. Current size is GB: {} .".format(row["ADM0_NAME"], (out_image.nbytes) / np.power(10.0,9)))
                        
                        # aggregated_value = split_and_aggregate(out_image, out_transform, pixel_size, width, height)

                    # else:
                        # aggregated_value = aggregate_one_region(out_image, out_transform, pixel_size, 0, 0, width, height)

                        # out_image = None # Remove the array.
                        # out_transform = None     

                    """OPTION2 : split the region into a grid (this is the most optimal)"""
                        
                    # create a grid of the territory with the coresponding EPSG
                    grid = get_geometry_grid(geodf_row, "EPSG:4326")
                    print("grid memory usage {}".format(sys.getsizeof(grid)))
                    # adjust the grid to the shape of the territory
                    region_grid = grid.overlay(geodf_row, how="intersection").to_crs(epsg='4326') # this operation requires both inputs to be gdf.
                    print("region_grid memory usage {}".format(sys.getsizeof(region_grid)))
                    # region_grid.to_file('region_grid.shp') # check memory for testing
                    # iterate over each tile and accumulate the value
                    total_aggregated_value = 0
                    total_tiles = int(len(region_grid))
                    for row_index, tile_row in region_grid.iterrows():
                        # print a process index.
                        print("{} out of {} of country {}".format(row_index + 1, total_tiles, row["ADM0_NAME"]))
                        geo_tile_row = gpd.GeoSeries(tile_row['geometry'])
                        # repeat the masking process with the tile.
                        out_image, out_transform = rasterio.mask.mask(raster_file, geo_tile_row, crop=True)
                        # print("out_image memory usage {}".format(out_image.nbytes / np.power(10.0,9))) # check memory for testing
                        # Obtain the number of tiles in both directions.
                        height = out_image.shape[1]
                        width  = out_image.shape[2]
                        #calculate the aggregated value.
                        aggregated_value = aggregate_one_region(out_image, out_transform, pixel_size, 0, 0, width, height)
                        # print("aggregated value memory usage {}".format(sys.getsizeof(aggregated_value))) # check memory for testing
                        # accumulate the results.
                        total_aggregated_value += aggregated_value 
                        # clean memory
                        out_image = None # this cleans the memory to 16 bits
                        out_transform = None

                        #force cleaning the memory
                        gc.collect()
                        
                except Exception as e:
                    # In case there is an error on the process, a value of -9999.0 will be appended
                    print("the country {} with index {} has errors: {}".format(row["ADM0_NAME"], row["OBJECTID"], e) )
                    error_countries_id.append(row["OBJECTID"])
                    aggregated_value = -9999.0 

                
                # Add the aggregated stock to the list.
                aggregated_value_list.append(total_aggregated_value)

                print("the country {} with index {} is finished with total carbon of: {}".format(row["ADM0_NAME"], row["OBJECTID"], total_aggregated_value))
                
                """this part is to create additional internal temporal results"""
                # country_counter += 1
                # if country_counter_iterator < 26: # + 1
                #     country_counter_iterator += 1
                # else:
                # if not landcover_class:
                #     aggregated_observable = pd.DataFrame(row["ADM0_NAME"], aggregated_value, columns = ["country", file_year])
                #     print(aggregated_observable.head())
                #     # Export the temporary results from curent year.
                #     aggregated_observable.to_csv(temp_export_path + "_" + row["ADM0_NAME"] + "_" + str(file_year) + ".csv")
                # else:
                #     aggregated_observable = pd.DataFrame(row["ADM0_NAME"], aggregated_value, columns = ["country", file_year + "_" + landcover_class])
                #     aggregated_observable.to_csv(temp_export_path + "_" + row["ADM0_NAME"] + "_" + str(file_year) + "_" + str(landcover_class) + ".csv")

                    # country_counter_iterator = 1

        print("Finished calculating year {}.".format(file_year))
        print("countries id with error: ", error_countries_id)

        # Transform the list to a DataFrame using the year as header.
        if not landcover_class:
            aggregated_observable = pd.DataFrame(aggregated_value_list, columns = [file_year])
            # Export the temporary results from curent year.
            aggregated_observable.to_csv(temp_export_path + "_" + str(file_year) + ".csv")
        else:
            aggregated_observable = pd.DataFrame(aggregated_value_list, columns = [file_year + "_" + landcover_class])
            aggregated_observable.to_csv(temp_export_path + "_" + str(file_year) + "_" + str(landcover_class) + ".csv")
            

        # Merge this year's results with the final, multi-year DataFrame.
        aggregated_df = pd.merge(aggregated_df, aggregated_observable, how='outer', left_index = True, right_index=True)

    aggregated_df.to_csv(temp_export_path + "_total_" + str(file_year) + ".csv")

    return aggregated_df

<>:40: SyntaxWarning: "is" with a literal. Did you mean "=="?
<>:40: SyntaxWarning: "is" with a literal. Did you mean "=="?
C:\Users\ruben.crespo\AppData\Local\Temp\ipykernel_10768\32763886.py:40: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if platform.system() is "Windows":


In [5]:
raster_list = get_raster_data(raster_directory)
region_polygons = load_region_polygons(region_polygons_file)

In [5]:
raster_list[0]

'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2010_global_300m_10.tif'

In [6]:
year_list = [2018]
  # Full path for temporary exports.
temp_export_path = temp_export_dir + observable_name
argument_list = parallel_argument_list(year_list, raster_list, region_polygons, temp_export_path)

In [9]:
argument_list[0]

(['C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_10.tif',
  'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_100.tif',
  'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_11.tif',
  'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_110.tif',
  'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_12.tif',
  'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_120.tif',
  'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_121.tif',
  'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_122.tif',
  'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_130.tif',
  'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_140.tif',
  'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_150.tif',
  'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_151.tif',
  'C:\\Users\\ruben

In [7]:
for i in argument_list[0]:
    print(i)

['C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_10.tif', 'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_100.tif', 'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_11.tif', 'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_110.tif', 'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_12.tif', 'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_120.tif', 'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_121.tif', 'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_122.tif', 'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_130.tif', 'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_140.tif', 'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_150.tif', 'C:\\Users\\ruben.crespo\\Downloads\\vcs_layers\\vcs_2018_global_300m_151.tif', 'C:\\Users\\ruben.crespo\\Downloads\\vcs_l

In [11]:
if __name__ == "__main__":
    print("Loading data.")
    raster_list = get_raster_data(raster_directory)
    region_polygons = load_region_polygons(region_polygons_file)
    print("Data loaded succesfully.")
    
    # Full path for temporary exports.
    temp_export_path = temp_export_dir + observable_name
    
    print("Starting aggregation process.")
    #list of years to compute
    # year_list = range(2001,2021) # always +1
    year_list = [2015, 2016, 2017]
    argument_list = parallel_argument_list(year_list, raster_list, region_polygons, temp_export_path)
    with Pool(processes= 4) as pool:
        print("Starting Pool.")
        # result = pool.starmap(aggregate_density_observable,argument_list)
        result = pool.starmap(aggregate_density_observable,argument_list)
        print(result)
        
    print("Aggregation finished.")
    # aggregated_observable = aggregate_density_observable(raster_list, region_polygons, temp_export_path)


    print("Exporting the aggregated dataset.")
    # export_to_csv(region_polygons, result, export_path)
    print("Done.")

Loading data.
Data loaded succesfully.
Starting aggregation process.
Starting Pool.
